# GPU-Fuzzy Trading Pipeline in VS Code Colab

This notebook is meant to run on a Colab server connected from the VS Code Colab extension.

Before running the cells:
- Connect the notebook to a Colab server.
- Either upload the entire `trading_platform/` folder to that server using Explorer > Upload to Colab, or let the bootstrap cell clone the private GitHub repo.
- If the folder lands somewhere else, set `PROJECT_ROOT` in the bootstrap cell to the uploaded path.
- If `train_new.csv` and `test_new.csv` live on Google Drive, the bootstrap cell copies them to `/content/trading_platform_data/` (local disk). A later cell causally enriches those raw tapes and points the pipeline at the enriched files.

The bootstrap cell mounts Google Drive when needed; the run cell writes outputs to `/content/trading_platform_outputs/` and syncs to Drive after success.

This notebook is the Colab T4 path: Phase 2 stays on the JAX GPU, uses a safe batch size of 64 and scan unroll of 16, and does not use the local RTX 4050 large-window CPU route. The local command-line path keeps its separate hybrid CPU/GPU policy.

Run the cells in order. If the dependency cell reports that JAX was already imported or the backend is not `gpu`, restart the Colab runtime and rerun from Cell 1.

If a previous Colab run patched `gpu_fuzzy_trader/evolution/numba_ops.py` on the server, re-clone the repo or delete `/content/trading_platform` and restart the kernel before rerunning.


In [ ]:
import os
from getpass import getpass

github_token = os.environ.get("GITHUB_TOKEN", "").strip()
if not github_token:
    github_token = getpass("GitHub classic PAT (optional, press Enter to skip): " ).strip()
if github_token:
    os.environ["GITHUB_TOKEN"] = github_token


In [ ]:
from base64 import b64encode
from pathlib import Path
from urllib.parse import urlsplit
import os
import shutil
import subprocess
import sys


PROJECT_ROOT = os.environ.get("PROJECT_ROOT", "").strip() or None
DEFAULT_GITHUB_REPO_URL = "https://github.com/m-danaee/trading_platform.git"
GITHUB_REPO_URL = os.environ.get("GITHUB_REPO_URL", DEFAULT_GITHUB_REPO_URL).strip() or DEFAULT_GITHUB_REPO_URL
GITHUB_BRANCH = os.environ.get("GITHUB_BRANCH", "main").strip() or "main"
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "").strip() or None
GITHUB_CLONE_DIR = Path(os.environ.get("GITHUB_CLONE_DIR", "/content/trading_platform")).expanduser()
os.environ["DATA_ROOT"] = "/content/drive/MyDrive"
os.environ["TRAIN_CSV_PATH"] = "/content/drive/MyDrive/train_new.csv"
os.environ["TEST_CSV_PATH"] = "/content/drive/MyDrive/test_new.csv"
DATA_ROOT = os.environ.get("DATA_ROOT", "").strip() or None
TRAIN_CSV_PATH = os.environ.get("TRAIN_CSV_PATH", "").strip() or None
TEST_CSV_PATH = os.environ.get("TEST_CSV_PATH", "").strip() or None


def _is_project_root(path: Path) -> bool:
    return (path / "gpu_fuzzy_trader").is_dir() and (path / "requirements.txt").is_file()


def _discover_project_root() -> Path | None:
    if PROJECT_ROOT is not None:
        candidate = Path(PROJECT_ROOT).expanduser()
        if _is_project_root(candidate):
            return candidate.resolve()

    for candidate in [Path.cwd(), Path("/content/trading_platform"), Path("/content")]:
        if _is_project_root(candidate):
            return candidate.resolve()

    content_root = Path("/content")
    if content_root.exists():
        for requirements_file in content_root.rglob("requirements.txt"):
            candidate = requirements_file.parent
            if _is_project_root(candidate):
                return candidate.resolve()

    return None


def _repo_url_has_credentials(repo_url: str) -> bool:
    parsed = urlsplit(repo_url)
    return bool(parsed.username or parsed.password)


def _ensure_drive_mounted() -> Path | None:
    drive_root = Path("/content/drive")
    if drive_root.exists():
        return drive_root

    try:
        from google.colab import drive
    except Exception:
        return None

    try:
        drive.mount("/content/drive", force_remount=False)
    except Exception:
        return None

    return drive_root if drive_root.exists() else None


def _path_has_csv_pair(root: Path) -> bool:
    return (root / "train_new.csv").is_file() and (root / "test_new.csv").is_file()


def _discover_dataset_paths(project_root: Path) -> tuple[Path, Path] | None:
    explicit_train = Path(TRAIN_CSV_PATH).expanduser() if TRAIN_CSV_PATH else None
    explicit_test = Path(TEST_CSV_PATH).expanduser() if TEST_CSV_PATH else None
    if explicit_train and explicit_test and explicit_train.is_file() and explicit_test.is_file():
        return explicit_train.resolve(), explicit_test.resolve()

    if DATA_ROOT:
        candidate = Path(DATA_ROOT).expanduser()
        if _path_has_csv_pair(candidate):
            return (candidate / "train_new.csv").resolve(), (candidate / "test_new.csv").resolve()

    for candidate in [
        project_root / "data",
        project_root,
        Path("/content/trading_platform/data"),
        Path("/content/trading_platform"),
    ]:
        if _path_has_csv_pair(candidate):
            return (candidate / "train_new.csv").resolve(), (candidate / "test_new.csv").resolve()

    drive_root = _ensure_drive_mounted()
    if drive_root is not None:
        drive_candidates = [
            drive_root / "MyDrive",
            drive_root / "My Drive",
            drive_root / "Shareddrives",
            drive_root / "Shared drives",
            drive_root,
        ]
        for candidate in drive_candidates:
            if _path_has_csv_pair(candidate):
                return (candidate / "train_new.csv").resolve(), (candidate / "test_new.csv").resolve()

        for candidate in drive_candidates:
            if not candidate.exists():
                continue
            for train_file in candidate.rglob("train_new.csv"):
                candidate_dir = train_file.parent
                if (candidate_dir / "test_new.csv").is_file():
                    return train_file.resolve(), (candidate_dir / "test_new.csv").resolve()

    content_root = Path("/content")
    if content_root.exists():
        for train_file in content_root.rglob("train_new.csv"):
            candidate = train_file.parent
            if (candidate / "test_new.csv").is_file():
                return train_file.resolve(), (candidate / "test_new.csv").resolve()

    return None


def _ensure_repo_data_files(project_root: Path, train_csv_path: Path, test_csv_path: Path) -> None:
    data_dir = project_root / "data"
    data_dir.mkdir(parents=True, exist_ok=True)

    targets = {
        data_dir / "train_new.csv": train_csv_path,
        data_dir / "test_new.csv": test_csv_path,
    }

    for target, source in targets.items():
        if target.exists():
            continue
        try:
            target.symlink_to(source)
        except Exception:
            shutil.copy2(source, target)


LOCAL_DATA_DIR = Path(os.environ.get("LOCAL_DATA_DIR", "/content/trading_platform_data"))


def _stage_csv_to_local(source: Path, filename: str) -> Path:
    """Copy Drive-hosted CSVs to local Colab disk for faster pipeline I/O."""
    source = source.resolve()
    if not str(source).startswith("/content/drive"):
        return source
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    target = (LOCAL_DATA_DIR / filename).resolve()
    if not target.is_file() or target.stat().st_size != source.stat().st_size:
        shutil.copy2(source, target)
    return target


def _repair_stale_numba_ops(project_root: Path) -> str:
    """Remove broken JIT empty branches left by older Colab notebook patches."""
    numba_ops_path = project_root / "gpu_fuzzy_trader" / "evolution" / "numba_ops.py"
    if not numba_ops_path.is_file():
        return "numba_ops.py not found"

    text = numba_ops_path.read_text(encoding="utf-8")
    original = text
    for block in (
        "        if n == 0:\n            return [[]]\n\n",
        "        if n == 0:\n            return [[-1]]\n\n",
        "        if n == 0:\n            return [[]]\n",
        "        if n == 0:\n            return [[-1]]\n",
    ):
        text = text.replace(block, "")

    wrapper_guard = (
        "    if obj.shape[0] == 0:\n"
        "        return [[]]\n"
    )
    if wrapper_guard not in text and "def non_dominated_sort" in text:
        needle = "    obj = np.asarray(objectives, dtype=np.float64)\n"
        if needle in text:
            text = text.replace(
                needle,
                needle + wrapper_guard,
                1,
            )

    if text != original:
        numba_ops_path.write_text(text, encoding="utf-8")
        return "repaired stale JIT empty branch in numba_ops.py"

    jit_parts = text.split("def _non_dominated_sort_numba", 1)
    if len(jit_parts) > 1:
        jit_body = jit_parts[1].split("\n    @", 1)[0].split("\n    def ", 1)[0]
        if "if n == 0:" in jit_body:
            raise RuntimeError(
                "numba_ops.py still has a Numba-incompatible empty branch. "
                "Delete /content/trading_platform and rerun bootstrap, or push the latest repo and git pull."
            )

    return "numba_ops.py ok"


def _try_git_pull(project_root: Path) -> str:
    git_dir = project_root / ".git"
    if not git_dir.is_dir():
        return "not a git repo"
    result = subprocess.run(
        ["git", "-C", str(project_root), "pull", "--ff-only"],
        capture_output=True,
        text=True,
    )
    if result.returncode == 0:
        return (result.stdout or "").strip() or "git pull ok"
    return f"git pull skipped: {(result.stderr or result.stdout or '').strip()}"


def _looks_like_auth_failure(output: str) -> bool:
    text = output.lower()
    return any(marker in text for marker in [
        'authentication failed',
        'could not read username',
        'repository not found',
        'terminal prompts disabled',
        'support for password authentication was removed',
        'access denied',
    ])


def _basic_auth_header(token: str) -> str:
    token_bytes = f'x-access-token:{token}'.encode('utf-8')
    encoded = b64encode(token_bytes).decode('ascii')
    return f'AUTHORIZATION: basic {encoded}'


def _run_git_clone(repo_url: str, target_dir: Path, token: str | None = None) -> subprocess.CompletedProcess[str]:
    clone_cmd = [
        'git',
        'clone',
        '--depth',
        '1',
        '--branch',
        GITHUB_BRANCH,
        repo_url,
        str(target_dir),
    ]
    clone_env = os.environ.copy()
    clone_env['GIT_TERMINAL_PROMPT'] = '0'
    clone_kwargs = {'capture_output': True, 'text': True, 'env': clone_env}
    if token:
        clone_cmd = [
            'git',
            '-c',
            f'http.extraheader={_basic_auth_header(token)}',
            'clone',
            '--depth',
            '1',
            '--branch',
            GITHUB_BRANCH,
            repo_url,
            str(target_dir),
        ]
    return subprocess.run(clone_cmd, **clone_kwargs)


def _clone_private_repo(repo_url: str, target_dir: Path) -> Path:

    if not repo_url:
        raise ValueError("GITHUB_REPO_URL is empty.")

    if target_dir.exists():
        if _is_project_root(target_dir):
            _try_git_pull(target_dir)
            return target_dir.resolve()
        if target_dir.is_dir() and any(target_dir.iterdir()):
            raise FileExistsError(
                f"{target_dir} already exists and is not the project root. "
                "Change GITHUB_CLONE_DIR or remove the conflicting directory."
            )
    target_dir.mkdir(parents=True, exist_ok=True)

    clone_result = _run_git_clone(repo_url, target_dir, GITHUB_TOKEN)
    if clone_result.returncode != 0:
        combined_output = (clone_result.stdout or '') + '\n' + (clone_result.stderr or '')
        hint = ""
        if _looks_like_auth_failure(combined_output):
            hint = (
                'This looks like a GitHub auth problem. For a classic PAT, use a token '
                'with repo read access, store it in Cell 1, and make sure the repository URL is correct.'
            )
        raise RuntimeError(
            'Git clone failed.\n'
            f'{combined_output.strip() or "(no git output)"}\n'
            f'{hint}'
        )

    return target_dir.resolve()


project_root = _discover_project_root()
if project_root is None and GITHUB_REPO_URL:
    project_root = _clone_private_repo(GITHUB_REPO_URL, GITHUB_CLONE_DIR)

if project_root is None:
    content_entries = []
    content_root = Path("/content")
    if content_root.exists():
        for path in sorted(content_root.iterdir()):
            content_entries.append(f"{path.name}/" if path.is_dir() else path.name)
    raise FileNotFoundError(
        "Could not find the project folder on the Colab server.\n"
        "Either upload the entire trading_platform/ folder, or set GITHUB_REPO_URL "
        "(and store your classic PAT in Cell 1 if the repo is private) and rerun it.\n"
        f"/content currently contains: {', '.join(content_entries) if content_entries else '(empty or unavailable)'}"
    )

resolved_dataset_paths = _discover_dataset_paths(project_root)
if resolved_dataset_paths is None:
    raise FileNotFoundError(
        "Could not find train_new.csv and test_new.csv.\n"
        "Either set DATA_ROOT or TRAIN_CSV_PATH / TEST_CSV_PATH to the Google Drive location, "
        "or upload both files to the project data/ directory and rerun Cell 2."
    )

train_csv_path, test_csv_path = resolved_dataset_paths
train_csv_path = _stage_csv_to_local(train_csv_path, "train_new.csv")
test_csv_path = _stage_csv_to_local(test_csv_path, "test_new.csv")
_ensure_repo_data_files(project_root, train_csv_path, test_csv_path)
numba_ops_status = _repair_stale_numba_ops(project_root)
if (project_root / ".git").is_dir():
    git_pull_status = _try_git_pull(project_root)
    numba_ops_status = _repair_stale_numba_ops(project_root)
else:
    git_pull_status = "not a git repo"
os.environ["TRAIN_CSV_PATH"] = str(train_csv_path)
os.environ["TEST_CSV_PATH"] = str(test_csv_path)
if train_csv_path.parent == test_csv_path.parent:
    os.environ["DATA_ROOT"] = str(train_csv_path.parent)

PROJECT_ROOT = project_root
os.environ["PROJECT_ROOT"] = str(project_root)
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

existing_pythonpath = os.environ.get("PYTHONPATH")
os.environ["PYTHONPATH"] = (
    str(project_root)
    if not existing_pythonpath
    else f"{project_root}{os.pathsep}{existing_pythonpath}"
)

print(f"Project root: {project_root}")
print(f"Python: {sys.executable}")
print(f"GitHub repo URL configured: {bool(GITHUB_REPO_URL)}")
print(f"GitHub URL has embedded credentials: {_repo_url_has_credentials(GITHUB_REPO_URL)}")
print(f"Train CSV: {train_csv_path}")
print(f"Test CSV: {test_csv_path}")
print(f"Numba ops: {numba_ops_status}")
print(f"Git sync: {git_pull_status}")
print(f"Repo data links ready: {(project_root / 'data' / 'train_new.csv').exists() and (project_root / 'data' / 'test_new.csv').exists()}")
print(f"Upload detected under /content: {str(project_root).startswith('/content')}")


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

PIP = [sys.executable, "-m", "pip", "install", "-q"]

_in_colab = (
    os.environ.get("COLAB_RELEASE_TAG") is not None
    or Path("/content").is_dir()
    or importlib.util.find_spec("google.colab") is not None
)

# Pin set aligned with requirements.txt and tested in local .venv.
# On Colab T4 (CUDA 12.x), install JAX via jax[cuda12] *after* core deps so
# generic CPU jaxlib wheels from requirements.txt do not win.
COLAB_CORE_PINS = [
    "numpy>=2.0,<2.4",
    "pandas>=2.0,<3",
    "numba>=0.60,<0.66",
    "scikit-learn>=1.3,<2",
    "matplotlib>=3.7,<4",
    "pyarrow>=14.0",
    "optuna>=3.5.0,<5",
]
COLAB_EVOX = "evox==1.3.0"
COLAB_JAX = "jax[cuda12]==0.10.1"
# EvoX NSGA-III selection uses PyTorch tensors; CPU wheel is enough and smaller.
COLAB_TORCH_CPU = (
    "torch==2.6.0",
    "--index-url",
    "https://download.pytorch.org/whl/cpu",
)

if _in_colab:
    print("Colab T4: installing pinned GPU stack (CUDA 12 + JAX 0.10.1 + EvoX)...")
    subprocess.check_call(PIP + ["-U", "pip", "wheel"])
    subprocess.check_call(PIP + COLAB_CORE_PINS)
    subprocess.check_call(PIP + list(COLAB_TORCH_CPU))
    subprocess.check_call(PIP + [COLAB_EVOX])
    # Install GPU JAX last to override Colab/previous CPU jaxlib wheels.
    subprocess.check_call(PIP + ["-U", COLAB_JAX])

    # Configure the T4 path before the first JAX import. The environment is
    # inherited by the pipeline subprocess launched in the run cell.
    COLAB_GPU_BATCH_SIZE = int(os.environ.get("COLAB_GPU_BATCH_SIZE", "64"))
    if COLAB_GPU_BATCH_SIZE < 1:
        raise ValueError("COLAB_GPU_BATCH_SIZE must be positive")
    os.environ["PHASE2_GPU_BATCH_SIZE"] = str(COLAB_GPU_BATCH_SIZE)
    os.environ["PHASE2_GPU_BATCH_SIZE_AUTO"] = "false"
    # Keep host RAM available for XLA compilation and Python-side evolution.
    os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.70")
    from gpu_fuzzy_trader._jax_env import configure_jax_env

    configure_jax_env()
    import jax

    print(
        f"JAX {jax.__version__} | backend={jax.default_backend()} | "
        f"devices={jax.devices()}"
    )
    if jax.default_backend() != "gpu":
        raise RuntimeError(
            "JAX is not using the GPU. Choose Runtime > Change runtime type > "
            "T4 GPU, then Runtime > Restart session and rerun from Cell 1."
        )
else:
    print("Installing dependencies from requirements.txt ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
    # Local NVIDIA GPU (e.g. RTX 4050 on WSL): install the CUDA 13 extra
    # after the CPU-compatible base; it keeps JAX and jaxlib at 0.10.1.
    LOCAL_GPU_PINS = [
        "jax[cuda13]==0.10.1",
    ]
    try:
        subprocess.check_call(PIP + ["-U"] + LOCAL_GPU_PINS)
        import jax
        print(
            f"JAX {jax.__version__} | backend={jax.default_backend()} | "
            f"devices={jax.devices()}"
        )
    except Exception as exc:
        print(f"GPU JAX install skipped ({exc}); Phase 2 may use CPU backtests.")


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# TRAIN_CSV_PATH and TEST_CSV_PATH still identify the staged raw tapes at
# this point. Enrich them before importing gpu_fuzzy_trader.config in the
# next cell so all pipeline modules resolve the causal HWC/MWC/LWC inputs.
raw_train = Path(os.environ["TRAIN_CSV_PATH"]).resolve()
raw_test = Path(os.environ["TEST_CSV_PATH"]).resolve()
raw_forward_value = (
    os.environ.get("RAW_FORWARD_CSV_PATH", "").strip()
    or os.environ.get("FORWARD_CSV_PATH", "").strip()
)
raw_forward = Path(raw_forward_value).expanduser().resolve() if raw_forward_value else None
if raw_forward is not None and not raw_forward.is_file():
    raise FileNotFoundError(f"Configured raw forward tape not found: {raw_forward}")

# Keep enrichment on local Colab disk. When raw train/test share the staged
# directory, config's default manifest path resolves to this same folder.
enriched_dir = raw_train.parent / "enriched"
enriched_dir.mkdir(parents=True, exist_ok=True)
enrich_cmd = [
    sys.executable,
    "-m",
    "gpu_fuzzy_trader.data.trend_context",
    "--train",
    str(raw_train),
    "--test",
    str(raw_test),
    "--out-dir",
    str(enriched_dir),
]
if raw_forward is not None:
    enrich_cmd.extend(["--forward", str(raw_forward)])

print("Generating causal HWC/MWC/LWC tapes:", " ".join(enrich_cmd), flush=True)
subprocess.check_call(enrich_cmd, cwd=str(PROJECT_ROOT), env=os.environ.copy())

enriched_train = enriched_dir / "train_new_hwc_mwc_lwc.csv"
enriched_test = enriched_dir / "test_new_hwc_mwc_lwc.csv"
enriched_forward = enriched_dir / "forward_hwc_mwc_lwc.csv"
manifest_path = enriched_dir / "trend_context_manifest.json"
for required in (enriched_train, enriched_test, manifest_path):
    if not required.is_file():
        raise FileNotFoundError(f"Trend-context enrichment did not produce {required}")

os.environ["TRAIN_CSV_PATH"] = str(enriched_train)
os.environ["TEST_CSV_PATH"] = str(enriched_test)
if raw_forward is not None:
    if not enriched_forward.is_file():
        raise FileNotFoundError(f"Forward enrichment did not produce {enriched_forward}")
    os.environ["FORWARD_CSV_PATH"] = str(enriched_forward)
else:
    os.environ.pop("FORWARD_CSV_PATH", None)

print(f"Enriched train: {enriched_train}")
print(f"Enriched test: {enriched_test}")
print(f"Context manifest: {manifest_path}")
print(f"Enriched forward: {enriched_forward if raw_forward is not None else '(not configured)'}")


In [ ]:
import importlib.metadata as _im
import os
import jax
from pathlib import Path

from gpu_fuzzy_trader import config as cfg
from gpu_fuzzy_trader.run_pipeline import Pipeline_Orchestrator

# The dependency cell configured XLA before importing JAX. Colab-specific
# route and scan defaults are also applied in the worker subprocess.

print("Import OK")
print(f"Enriched TRAIN_CSV_PATH: {Path(cfg.TRAIN_CSV_PATH).resolve()}")
print(f"Enriched TEST_CSV_PATH: {Path(cfg.TEST_CSV_PATH).resolve()}")
print(f"Context manifest: {Path(cfg.ENRICHED_MANIFEST_PATH).resolve()}")
print(f"JAX backend: {jax.default_backend()} | devices: {jax.devices()}")
for _pkg in ("evox", "optuna", "numba", "torch", "jax"):
    try:
        print(f"{_pkg}: {_im.version(_pkg)}")
    except _im.PackageNotFoundError:
        print(f"{_pkg}: not installed")

try:
    import torch
    from evox.operators.sampling.uniform import uniform_sampling
    from evox.operators.selection.non_dominate import non_dominate_rank

    print("EvoX NSGA-III ready")
except ImportError as _evox_exc:
    raise RuntimeError(
        "EvoX operators failed to import — Phase 2 will fall back to NSGA-II. "
        f"Re-run the install cell or upgrade evox/torch. ({_evox_exc})"
    ) from _evox_exc

from gpu_fuzzy_trader._gpu_runtime import (
    detect_gpu_vram_gb,
    resolve_phase2_gpu_batch_size,
)

_vram = detect_gpu_vram_gb()
_vram_s = f"{_vram:.1f} GiB" if _vram is not None else "unknown"
_jax_cache = os.environ.get("JAX_COMPILATION_CACHE_DIR", "(not set)")
print(
    f"Colab GPU defaults: enabled={cfg.is_colab_runtime()} | "
    f"phase2_gpu={cfg.PHASE2_USE_GPU} | "
    f"batch_auto_env={os.environ.get('PHASE2_GPU_BATCH_SIZE_AUTO', '(default)')} | "
    f"large_window_cpu_route={cfg.PHASE2_GPU_CPU_ROUTE_LARGE_DATA} | "
    f"jax_cache={_jax_cache}"
)
print(
    f"PHASE2_USE_GPU={cfg.PHASE2_USE_GPU} | "
    f"batch_size={resolve_phase2_gpu_batch_size()} "
    f"(config_default={cfg.PHASE2_GPU_BATCH_SIZE}) | "
    f"scan_unroll={cfg.PHASE2_SCAN_UNROLL} | vram={_vram_s}"
)
if jax.default_backend() != "gpu" and cfg.PHASE2_USE_GPU:
    print(
        "WARNING: PHASE2_USE_GPU=True but JAX backend is not GPU — "
        "Phase 2 will fall back to CPU backtests (much slower)."
    )

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import jax

# The dependency cell set the effective T4 batch/XLA environment before
# JAX import; keep it unchanged so the worker subprocess inherits it.

# Run controls:
#   RUN_PHASE : None = full pipeline, or 1, 2, or 5 for a production
#               partial run. Inputs 3 and 4 are retained only as
#               compatibility aliases for the canonical RB Governor.
#   RESUME    : skip already-completed outputs.
RUN_PHASE = None
RESUME = False
USE_LOCAL_SCRATCH = True
LOCAL_OUTPUT_DIR = Path("/content/trading_platform_outputs")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/trading_platform/outputs")
OUTPUT_DIR = str(LOCAL_OUTPUT_DIR if USE_LOCAL_SCRATCH else DRIVE_OUTPUT_DIR)
PHASE2_ARCHIVE_SRC = Path(PROJECT_ROOT) / "phase2_rule_archive"
PHASE2_ARCHIVE_DST = Path("/content/drive/MyDrive/trading_platform/phase2_rule_archive")

print(f"JAX backend: {jax.default_backend()} | devices: {jax.devices()}", flush=True)

if USE_LOCAL_SCRATCH:
    LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if RESUME and DRIVE_OUTPUT_DIR.is_dir():
        shutil.copytree(DRIVE_OUTPUT_DIR, LOCAL_OUTPUT_DIR, dirs_exist_ok=True)
        print(f"Resume: copied prior outputs from Drive -> {LOCAL_OUTPUT_DIR}", flush=True)

pipeline_cmd = [
    sys.executable,
    "-u",
    "-m",
    "gpu_fuzzy_trader.run_pipeline",
    "--output",
    OUTPUT_DIR,
]
if RUN_PHASE is not None:
    pipeline_cmd.extend(["--phase", str(RUN_PHASE)])
if RESUME:
    pipeline_cmd.append("--resume")

print("Running:", " ".join(pipeline_cmd), flush=True)
t0 = time.perf_counter()

process = subprocess.Popen(
    pipeline_cmd,
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)

assert process.stdout is not None
for line in iter(process.stdout.readline, ""):
    print(line, end="", flush=True)

return_code = process.wait()
elapsed = time.perf_counter() - t0
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, pipeline_cmd)

print(f"\nPipeline finished in {elapsed:.1f}s", flush=True)

if USE_LOCAL_SCRATCH:
    _ensure_drive_mounted()
    if DRIVE_OUTPUT_DIR.parent.exists():
        DRIVE_OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(LOCAL_OUTPUT_DIR, DRIVE_OUTPUT_DIR, dirs_exist_ok=True)
        print(f"Outputs synced to Google Drive: {DRIVE_OUTPUT_DIR}", flush=True)

if PHASE2_ARCHIVE_SRC.is_dir():
    PHASE2_ARCHIVE_DST.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(PHASE2_ARCHIVE_SRC, PHASE2_ARCHIVE_DST, dirs_exist_ok=True)
    print(f"Phase 2 archive synced to Google Drive: {PHASE2_ARCHIVE_DST}", flush=True)
else:
    print(f"Phase 2 archive folder not found at: {PHASE2_ARCHIVE_SRC}", flush=True)

print("\nPipeline finished successfully.", flush=True)

After causal enrichment and the pipeline finish, artifacts are written to local `/content/trading_platform_outputs/` during the run, then synced to `MyDrive/trading_platform/outputs/` on Google Drive. `phase2_rule_archive/` is also synced to `MyDrive/trading_platform/phase2_rule_archive/`. Set `RAW_FORWARD_CSV_PATH` before the enrichment cell only when a strictly newer untouched forward tape is available.

The RB Governor is the canonical selection and risk pipeline and writes `outputs/long.json` and `outputs/short.json` in evaluator-compatible schema. Inputs `3` and `4` are reserved compatibility aliases only.

For partial reruns set `RESUME = True` and optionally `RUN_PHASE` to `1`, `2`, or `5`. Open `evaluator_v5.ipynb` and point it at the generated `outputs/long.json` and `outputs/short.json` files.